In [19]:
import sys
import json
import pickle
import random
import warnings
from tqdm import tqdm
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# openai
from openai import OpenAI, OpenAIError
import openai

import importlib
import random_walker.PageRank as pagerank
import random_walker.Evaluation as eval
# import recommender.Recommender as rcmd
import random_walker.ReverseRW as reverse
import llm_connector.Collector as collector
import datasets.MBA as mba_ds

In [20]:
# Suppress the specific DtypeWarning
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

## 1. data loading & preprocess

In [21]:
ds_path = '../data/interim/funnel_mba_format.csv'
[mba_df, user_ids, user_num, user_ids_kv, item_names, item_num, items_kv, G_user, G_item] = mba_ds.MBA_load_data(ds_path)

Loading MBA dataset from path:../data/interim/funnel_mba_format.csv
all nan eliminated
totally 4297 unique users
totally 3846 unique items


388023it [00:06, 57064.12it/s]


In [4]:
with open('../data/tri_graph_uidx2tidx_train.json', 'r') as f:
    tri_graph_uidx2tidx_train = json.load(f)
with open('../data/tri_graph_uidx2pidx.json', 'r') as f:
    tri_graph_uidx2pidx = json.load(f)
with open('../data/persona2idx.json', 'r') as f:
    persona2idx_whole = json.load(f)

In [5]:
persona2idx_whole_keys = list(persona2idx_whole.keys())
tri_graph_uidx2tidx_train = {int(k):v for k,v in tri_graph_uidx2tidx_train.items()}
tri_graph_uidx2pidx = {int(k):v for k,v in tri_graph_uidx2pidx.items()}
multi_labels = {}
for uidx,ps in tri_graph_uidx2pidx.items():
    multi_labels[user_ids[uidx]] = [persona2idx_whole_keys[pidx] for pidx in ps]

## 2. sample the prototype users & compute affinities with RevAff
## & generate personas for unlabeled users

In [15]:
all_uidxs = np.arange(user_num)
rep_uidxs = list(tri_graph_uidx2pidx.keys())

np.random.seed(191)
random.seed(191)

oracle = pagerank.Persona_Oracle(multi_labels, user_ids)
sample_scope = rep_uidxs

In [16]:
def make_app(sampled_GT, persona_probs, unlabeled_uidxs, test_scope_idxs, persona2idx_whole, persona2idx_whole_2,):
    multi_labels_approach = {} # {uidx: ps}
    tri_graph_uidx2pidx_approach = {} # remap from the 
    
    multi_labels_approach.update(sampled_GT)
    persona_predictions = np.argsort(persona_probs, axis=1) # small to large
    unlabeled_uidxs_2_prediction = eval.make_unlabeled_uidxs_2_prediction(unlabeled_uidxs)
    persona_approach_num = 5 #hp

    whole_personas_2 = list(persona2idx_whole_2.keys())
    for uidx in test_scope_idxs:
        assert uidx not in multi_labels_approach
        pred_idx = unlabeled_uidxs_2_prediction[uidx] # position of uidx in prediction matrix
        persona_idxs_rank = persona_predictions[pred_idx][::-1] # rerank from large to small
        ps_approach_pidxs = persona_idxs_rank[:persona_approach_num] # select largest 5
        ps_approach = [whole_personas_2[pidx] for pidx in ps_approach_pidxs]
        multi_labels_approach[uidx] = ps_approach

    for uidx, ps in multi_labels_approach.items():
        # ignore the persona label of the unrepresentable users
        assert 'Unrepresentable' not in ps # already all representative
        ps_idx = [persona2idx_whole[p] for p in ps] # use the original persona2idx_whole to make the transformation
        tri_graph_uidx2pidx_approach[uidx] = ps_idx

    tri_graph_uidx2pidx_approach = {int(k):[int(e) for e in v] for k,v in tri_graph_uidx2pidx_approach.items()} # reform the datatype
    return tri_graph_uidx2pidx_approach

In [18]:
# try with epsilon = 0.1, debias coefficient = 0.5, sample rate = 20%
for e in [0.1]:
    # 1. reverse sample
    np.random.seed(191)
    random.seed(191)
    sample_scope = rep_uidxs
    sampler_rev = reverse.Iterative_Sampler_rev(G_user_train, G_item_train, oracle, sample_scope, predefined_persona_list, e=e,)
    sampled_GT = sampler_rev.run(800, 100, seed_amount=100,)
    # 2. reverse prediction
    persona_probs, label_matrix, persona_list, persona2idx = reverse.persona_prediction_rev(e, G_user_train, sampled_GT, user_num, col_refine=0.5)
    persona2idx_whole_2 = eval.make_persona2idx_whole(persona2idx, predefined_persona_list)
    # make approximated rank
    labeled_uidxs = list(sampled_GT.keys())
    test_scope_idxs = np.setdiff1d(rep_uidxs, labeled_uidxs) # {representative users}/{labeled users}
    unlabeled_uidxs = np.setdiff1d(np.arange(user_num), labeled_uidxs) # {all users}/{labeled users}
    tri_graph_uidx2pidx_approach = make_app(sampled_GT, persona_probs, unlabeled_uidxs, test_scope_idxs, persona2idx_whole, persona2idx_whole_2)
    store_path = f'../data/gplr_res/tri_graph_uidx2pidx_app_e_{str(e)}.json'
    with open(store_path, 'w') as f:
        json.dump(tri_graph_uidx2pidx_approach, f)
    print(f'json file written for e={e}')

100%|████████████████████████████████████████████████████████████████████████████| 19/19 [00:00<00:00, 59.31it/s]

json file written for e=0.1
